# I-3D — Google Colab launcher

**One repository, one set of Python scripts.** Colab only clones [Apache0ne/I-3D](https://github.com/Apache0ne/I-3D), runs `scripts/colab.py`, and embeds the repository's [3D Studio](https://github.com/Apache0ne/I-3D/tree/main/viewer). **No model installation code or pipeline implementation is duplicated in this notebook.**

Pipeline: image → Hunyuan3D-2mini → Hi3D refinement → Hunyuan3D Paint 2.1 → TokenRig skinning → SOMA 30-joint bridge → START/TARGET 3D poses in the viewer → MotionBricks → animated GLB. Headless Blender is an internal export/rigging dependency, not a separate interface.

**Hardware:** NVIDIA CUDA 13.0 compiler and compatible GPU. A100 40GB is a reasonable development target; high-resolution Hi3D and full PBR generation may still exceed available GPU memory. CUDA-13 Hi3D, Paint and TokenRig installations remain experimental adaptations. **End-to-end inference has not been validated.** Model downloads are substantial, and pretrained weights retain their original licenses. The NVIDIA MotionBricks TensorRT environment retains its own upstream CUDA dependencies.


## 1. Clone the latest `main`

The notebook launches the Python files in GitHub. Re-running this cell fast-forwards an existing checkout instead of reusing old files.


In [ ]:
from pathlib import Path
import os, sys, subprocess
ROOT = Path("/content/I-3D")
WORK = ROOT / "outputs" / "character"

def run(*args):
    cmd = [str(a) for a in args]
    print("RUN:", " ".join(cmd), flush=True)
    # Inherit stdout and stderr so actual errors are visible in Colab.
    result = subprocess.run(cmd)
    if result.returncode:
        raise RuntimeError(f"Command failed ({result.returncode}): {' '.join(cmd)}")

if not ROOT.exists():
    run("git", "clone", "--branch", "main", "--single-branch",
        "https://github.com/Apache0ne/I-3D.git", ROOT)
else:
    run("git", "-C", ROOT, "fetch", "origin", "main")
    run("git", "-C", ROOT, "checkout", "main")
    run("git", "-C", ROOT, "merge", "--ff-only", "origin/main")
run("git", "-C", ROOT, "rev-parse", "--short", "HEAD")
WORK.mkdir(parents=True, exist_ok=True)


## 2. Install dependencies by running the repository's Python setup

The setup script installs system tools, separate Python environments and actual model dependencies, and downloads the original required model artifacts. It calls `install_hi3d_cuda13.py`, `install_paint_cuda13.py`, the upstream SkinTokens downloader, and the MotionBricks asset hydrator.

**If one stage fails:** fix the displayed error, then set `SETUP_STAGE` to only that stage and rerun. Successful stages need not be reinstalled.


In [ ]:
SETUP_STAGE = "all"   # all / hi3d / paint / tokenrig / motion
MOTION_PRECISION = "fp16"  # fp16 or fp32
run(sys.executable, ROOT/"scripts"/"colab.py", "setup",
    "--stage", SETUP_STAGE, "--precision", MOTION_PRECISION)


## 3. Upload one full-body reference photograph

Choose an image where arms and legs are clearly visible, ideally in an A-pose or T-pose. The single source image is used through geometry, painting and rigging.


In [ ]:
from google.colab import files
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Upload exactly one PNG/JPEG/WEBP")
filename, data = next(iter(uploaded.items()))
ext = Path(filename).suffix.lower()
if ext not in (".png", ".jpg", ".jpeg", ".webp"):
    raise ValueError("Unsupported input format: " + ext)
IMAGE = WORK / ("original_input" + ext)
IMAGE.write_bytes(data)
print("Input:", IMAGE, "bytes:", IMAGE.stat().st_size)


## 4. Launch the interactive I-3D 3D Studio — inside Colab

Real-time GLB/PBR visualization, orbit/pan/zoom, stage comparisons, materials, wireframe, normals, skeleton and skin weights, animation timeline and scrubber, screenshots, recording, plus START/TARGET SOMA pose editing. The workspace detects output files as the model stages finish.

No separate Blender GUI or external viewer is needed.


In [ ]:
from google.colab import output
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from scripts.viewer_server import launch

PORT = 8765
if "_i3d_studio_server" not in globals():
    _i3d_studio_server = launch(WORK, port=PORT)
print("I-3D 3D Studio:")
output.serve_kernel_port_as_iframe(PORT, height=980)


## 5. Generate the textured, rigged 3D character

This runs the original Python model files via the repository orchestrator. The viewer can display intermediate GLB outputs. The output SOMA character is the **same** textured mesh fitted to the 30-joint skeleton.

Select the visual forward direction before generating. **Do not guess or silently mirror the axes.** The bridge rejects invalid coordinate transforms. On failure, the script keeps completed files to make reruns less expensive. Changing the input image or forward direction requires `FORCE_REBUILD=True`.


In [ ]:
CHARACTER_FORWARD = "+X"  # +X, -X, +Y, -Y in Blender Z-up
GEOMETRY_RESOLUTION = 512   # 512 / 1024 / 1536
HI3D_STEPS = 25
FORCE_REBUILD = False
args = [sys.executable, ROOT/"scripts"/"colab.py", "prepare",
        "--image", IMAGE, "--forward", CHARACTER_FORWARD,
        "--resolution", str(GEOMETRY_RESOLUTION),
        "--steps", str(HI3D_STEPS), "--output", WORK]
if FORCE_REBUILD:
    args.append("--force")
run(*args)
print("Ready to pose:", WORK / "04_soma_character.glb")


## 6. Author two **real 3D poses** in the Studio

In the viewer, select **SOMA 30-joint character**, edit its 3D bones and save **START** and **TARGET** poses. The viewer writes `viewer_poses.json` into the same Colab workspace. The Blender interface is not needed.

Reopen or refresh the Studio iframe below after generating the character; then save both poses before proceeding.


In [ ]:
output.serve_kernel_port_as_iframe(PORT, height=980)
print("Viewer pose file:", WORK/"viewer_poses.json")


## 7. MotionBricks animation and GLB export

The repository scripts convert the two viewer poses into proper Blender keyframes **headlessly**, export pose matrices, generate the 30 FPS MotionBricks transition, then apply it back onto the same textured character. This runs the existing `author_viewer_poses.py`, `export_two_poses.py`, `generate_motion.py` and `apply_motion_blender.py` scripts.


In [ ]:
run(sys.executable, ROOT/"scripts"/"colab.py", "animate",
    "--output", WORK, "--precision", MOTION_PRECISION)
print("Animated GLB:", WORK/"07_animated.glb")
output.serve_kernel_port_as_iframe(PORT, height=980)


## 8. Download all results

The repository collects geometry, PBR model, skinning, SOMA character, pose files, MotionBricks animation, diagnostics and editable `.blend` into one ZIP. No second viewer is needed.


In [ ]:
run(sys.executable, ROOT/"scripts"/"colab.py", "pack", "--output", WORK)
from google.colab import files
files.download(str(WORK/"I3D_output_bundle.zip"))


## Sources and limitations

- [All I-3D scripts](https://github.com/Apache0ne/I-3D/tree/main/scripts) — installers, orchestration, 3D rigging, animation and packaging.
- [Viewer source](https://github.com/Apache0ne/I-3D/tree/main/viewer) — interactive Colab Studio.
- [Hi3D original implementation](https://github.com/hi3d-official/Hi3Dv1.0), [Hunyuan3D-2](https://github.com/Tencent-Hunyuan/Hunyuan3D-2), [Hunyuan Paint 2.1](https://github.com/Tencent-Hunyuan/Hunyuan3D-2.1), [SkinTokens](https://github.com/VAST-AI-Research/SkinTokens), [SOMA-X](https://github.com/NVlabs/SOMA-X), [MotionBricks](https://github.com/NVlabs/motionbricks).

This simplified notebook is a **launcher, not a validated single-click production model**. Hi3D's upstream reference stack was CUDA 12.4; our CUDA-13 SpConv and FlashAttention wheels are unofficial, checksum-verified builds. Skinning quality, external model weights and TensorRT compatibility require actual inference on the chosen Colab runtime. Upstream code and model licenses apply independently.
